# OOP Deep Dive — Expanded Explanations and Examples
This notebook expands each concept from the original `1_oops_deep_dive.ipynb` with clearer explanations and extra examples.

## Classes: class vs instance attributes
Instance attributes belong to each object. Class attributes are shared across all instances — watch out for mutable shared state.

In [ ]:
class BankAccount:
    interest_rate = 0.02  # class attribute shared by all accounts

    def __init__(self, owner: str, balance: float = 0):
        self.owner = owner  # instance attribute
        self._balance = balance

    def deposit(self, amount: float):
        if amount <= 0:
            raise ValueError('deposit must be positive')
        self._balance += amount

    def withdraw(self, amount: float):
        if amount > self._balance:
            raise ValueError('insufficient funds')
        self._balance -= amount

    def apply_interest(self):
        self._balance += self._balance * self.interest_rate

    def __repr__(self):
        return f'BankAccount(owner={self.owner!r}, balance={self._balance:.2f})'

# Usage
a = BankAccount('Alice', 100)
b = BankAccount('Bob', 50)
BankAccount.interest_rate = 0.03  # change for all accounts
a.apply_interest()
b.apply_interest()
print(a, b)

## Mutable-default gotcha (shared mutable class attributes)
Avoid using mutable types (list/dict) as class attributes when you expect per-instance state.

In [ ]:
class BadCart:
    items = []  # shared across instances -> bug

    def add(self, item):
        self.items.append(item)

class GoodCart:
    def __init__(self):
        self.items = []  # distinct per instance

    def add(self, item):
        self.items.append(item)

c1, c2 = BadCart(), BadCart()
c1.add('apple')
print('BadCart c1.items:', c1.items)
print('BadCart c2.items:', c2.items)  # unexpected shared list

g1, g2 = GoodCart(), GoodCart()
g1.add('banana')
print('GoodCart g1.items:', g1.items)
print('GoodCart g2.items:', g2.items)  # separate lists

## `@classmethod` vs `@staticmethod`
Use `@classmethod` for alternative constructors or methods that need the class; use `@staticmethod` for utility functions that don't need class or instance access.

In [ ]:
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year):
        from datetime import date
        age = date.today().year - birth_year
        return cls(name, age)

    @staticmethod
    def is_adult(age):
        return age >= 18

p = Person.from_birth_year('Sam', 2000)
print(p.name, p.age, Person.is_adult(p.age))

## Inheritance, `super()` and MRO
`super()` lets subclasses extend parent behaviour. Python resolves methods by Method Resolution Order (MRO) — inspect with `.__mro__`.

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name
    def speak(self):
        raise NotImplementedError

class Dog(Animal):
    def __init__(self, name, breed):
        super().__init__(name)
        self.breed = breed
    def speak(self):
        return f'Woof! Im {self.name}, a {self.breed}'

class Cat(Animal):
    def speak(self):
        return f'Meow! Im {self.name}'

print(Dog('Rex', 'Lab').speak())
print(Cat('Luna').speak())

# Multiple inheritance example and MRO
class Flyable:
    def move(self):
        return 'flying'

class Swimmable:
    def move(self):
        return 'swimming'

class Duck(Flyable, Swimmable):
    pass

print('Duck moves:', Duck().move())
print('Duck MRO:', Duck.__mro__)

## Dunder (magic) methods — make objects behave like built-ins
Implement `__repr__`, `__str__`, `__add__`, `__len__`, `__eq__`, and optionally iteration/indexing for richer behavior.

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f'Vector({self.x}, {self.y})'

    def __str__(self):
        return f'({self.x}, {self.y})'

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __len__(self):
        return int((self.x**2 + self.y**2)**0.5)

    def __eq__(self, other):
        return isinstance(other, Vector) and self.x == other.x and self.y == other.y

    def __iter__(self):
        yield self.x
        yield self.y

v1 = Vector(1, 2)
v2 = Vector(3, 4)
print(v1 + v2)
print('len(v1)=', len(v1))
print('tuple(v1)=', tuple(v1))

## `__slots__` for memory efficiency
Use `__slots__` to avoid per-instance `__dict__` when creating many objects; note it restricts adding new attributes.

In [ ]:
class Point:
    __slots__ = ('x', 'y')
    def __init__(self, x, y):
        self.x, self.y = x, y

p = Point(1, 2)
print(p.x, p.y)

## `@property` and descriptors
Use `@property` to expose a computed or validated attribute while keeping a clean attribute-like API.

In [ ]:
class Temperature:
    def __init__(self, celsius: float):
        self._celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError('Below absolute zero!')
        self._celsius = value

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

t = Temperature(0)
print(t.fahrenheit)
t.celsius = 25
print(t.fahrenheit)

## Dataclasses — concise data containers
`@dataclass` reduces boilerplate. Use `field(default_factory=...)` for mutable defaults and `frozen=True` for immutability.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Book:
    title: str
    author: str
    tags: list[str] = field(default_factory=list)

b = Book('Fluent Python', 'Luciano')
b.tags.append('python')
print(b)

## Five-line Summary
- OOP groups state (attributes) and behavior (methods) into classes; prefer instance attributes for per-object state.
- Avoid mutable class attributes for per-instance data; use `default_factory` for dataclasses.
- Use `@classmethod` for alternate constructors and `@staticmethod` for utilities; `super()` and MRO control method lookup.
- Dunder methods make objects behave like built-ins; `__slots__` and `@dataclass` help performance and boilerplate.
- `@property` exposes controlled attributes with validation or computed values, improving API ergonomics.